In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,udf
from pyspark.sql.types import StringType, IntegerType
import time

In [2]:
spark = SparkSession.builder.appName("OptimizationDemo").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 06:00:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [12]:
df = spark.range(0, 10000000)
df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
| 10|
| 11|
| 12|
| 13|
| 14|
| 15|
| 16|
| 17|
| 18|
| 19|
+---+
only showing top 20 rows



In [4]:
optimized_df = df.filter(col("id") > 500) \
                 .filter(col("id") < 5000000) \
                 .select("id")

26/06/13 06:00:32 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
optimized_df.explain()

== Physical Plan ==
*(1) Filter ((id#0L > 500) AND (id#0L < 5000000))
+- *(1) Range (0, 10000000, step=1, splits=2)




In [ ]:


# GitHub Actions Test

start_time = time.time()

count_uncached = optimized_df.count()

end_time = time.time()

print(
    f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time - start_time, 4)} seconds"
)

1. Optimized Execution | Count: 4999499 | Time: 0.6301 seconds


In [ ]:

# GitHub Actions Test - Round 2
optimized_df.cache()

DataFrame[id: bigint]

In [8]:
optimized_df.count()

4999499

In [9]:
start_time = time.time()

count_cached = optimized_df.count()

end_time = time.time()

print(
    f"2. Cached Execution | Count: {count_cached} | Time: {round(end_time - start_time, 4)} seconds"
)

2. Cached Execution | Count: 4999499 | Time: 0.19 seconds


In [10]:
optimized_df.unpersist()

DataFrame[id: bigint]

In [13]:
data =[("Alice",25),("BOB",17),("charlie",35)]
df1 = spark.createDataFrame(data, ["Name", "Age"])
df1.show()

[Stage 12:>                                                         (0 + 1) / 1]

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    BOB| 17|
|charlie| 35|
+-------+---+



In [14]:
def categorize_age(age):
    if age >= 18:
        return "Adult"
    return "Minor"

In [15]:
age_category_udf = udf(categorize_age, StringType())

In [16]:
df_method1 = df1.withColumn(
    "Category",
    age_category_udf(col("Age"))
)
print("Method 1: Standard UDF via DataFrame API")
df_method1.show()


Method 1: Standard UDF via DataFrame API


+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    BOB| 17|   Minor|
|charlie| 35|   Adult|
+-------+---+--------+



In [17]:
def candrive(age):
    if age >= 18:
        return "Can drive"
    else:
        return "Cannot drive"




In [24]:
candrive_udf = udf(candrive, StringType())
df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
| 10|
| 11|
| 12|
| 13|
| 14|
| 15|
| 16|
| 17|
| 18|
| 19|
+---+
only showing top 20 rows



In [27]:
data = [
    ("Alice", 25),
    ("BOB", 17),
    ("Charlie", 35),
    ("David", 15)
]

df = spark.createDataFrame(data, ["Name", "Age"])

df.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    BOB| 17|
|Charlie| 35|
|  David| 15|
+-------+---+



In [29]:
def candrive(age):
    if age >= 18:
        return "Can Drive"
    else:
        return "Cannot Drive"
        

In [30]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

candrive_udf = udf(candrive, StringType())

In [31]:
df1 = df.withColumn(
    "Driving_Status",
    candrive_udf(col("Age"))
)

df1.show()

+-------+---+--------------+
|   Name|Age|Driving_Status|
+-------+---+--------------+
|  Alice| 25|     Can Drive|
|    BOB| 17|  Cannot Drive|
|Charlie| 35|     Can Drive|
|  David| 15|  Cannot Drive|
+-------+---+--------------+



In [34]:
spark.udf.register("sql_candrive", candrive, StringType())
df1.createOrReplaceTempView("people")

In [37]:
spark.sql("""
SELECT Name,
       Age,
       sql_candrive(Age) AS Driving_Status
FROM people
""").show()

+-------+---+--------------+
|   Name|Age|Driving_Status|
+-------+---+--------------+
|  Alice| 25|     Can Drive|
|    BOB| 17|  Cannot Drive|
|Charlie| 35|     Can Drive|
|  David| 15|  Cannot Drive|
+-------+---+--------------+

